### Fluxo desse modelo:

- Pega a tabela
- Preparação
    - Fazer o Categorical Encoding -- transformar as classes em seus respectivos valores numéricos
        - Ordinais -- "Saving accounts" e "Checking account"
        - Nominais -- "Purpose"
    - Retirar as linhas com mais que uma coluna vazia

- Lidando com os missing values e separando a base de teste
    - Separar os tabela em três: df_preenchido, df_col1_faltante e df_col2_faltante
    - Separamos parte da tabela com todos os atributos para o teste, mas mantemos a proporção de resultados da geral
    - Criamos um modelo de árvore de decisão para determinar a col1 faltante
    - Aplicamos esse modelo no df_col1_faltante
    - Criamos um modelo de árvore de decisão para determinar a col2 faltante
    - Aplicamos esse modelo no df_col2_faltante
    - Retiramos os dados que serão usados no teste da base geral

- Treino e teste do modelo

### Possíveis problemas:
- Proporção das bases 

In [7]:
import pandas as pd
import numpy as np

from sklearn import tree, set_config  # Arvore de decisão e plot tree, configurações
from sklearn.metrics import accuracy_score   # Acurácia
from sklearn.compose import ColumnTransformer # Tratar as colunas nominais de forma unificada
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, KBinsDiscretizer, StandardScaler # Transforma as colunas nominais, Knn para discretização, Normalização
from sklearn.model_selection import train_test_split  # Separar a parte de teste
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay  # Matriz de confusão
import matplotlib.pyplot as plt  # Plot na tabela

set_config(transform_output="pandas")

In [11]:
df = pd.read_csv('./class_german_credit.csv', sep=',')

# Sex
df['Sex'] = (df['Sex'] == 'male').astype(int) # female -> 0; male -> 1;

# Colunas nominais
preprocessor = ColumnTransformer(
    transformers=[
        ('nom_housing', OrdinalEncoder(categories=[['free', 'rent', 'own']]), ['Housing']),
        ('nom_saving',  OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan, categories=[['little', 'moderate', 'quite rich', 'rich']]), ['Saving accounts']),
        ('nom_checking',OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan, categories=[['little', 'moderate', 'rich']]), ['Checking account']),
        ('ord_purpose', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['Purpose'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

df = preprocessor.fit_transform(df)


# Risk
df['Risk'] = (df['Risk'] == 'good').astype(int) # bad -> 0; good -> 1;
df

,Housing,Saving accounts,Checking account,Purpose_business,Purpose_car,Purpose_domestic appliances,Purpose_education,Purpose_furniture/equipment,Purpose_radio/TV,Purpose_repairs,Purpose_vacation/others,Age,Sex,Job,Credit amount,Duration,Risk
0,2.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,67,1,2,1169,6,1
1,2.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,22,0,2,5951,48,0
2,2.0,0.0,NaN,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,49,1,1,2096,12,1
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,45,1,2,7882,42,1
4,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,53,1,2,4870,24,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,2.0,0.0,NaN,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,31,0,1,1736,12,1
996,2.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,40,1,3,3857,30,1
997,2.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,38,1,2,804,12,1
998,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,23,1,2,1845,45,0


In [49]:
_X = df.drop('Risk', axis=1)
y = df['Risk']

ct_age = ColumnTransformer([
        ('scaler', KBinsDiscretizer(n_bins=2, encode='ordinal', strategy='quantile'), ['Age'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

ct_credit = ColumnTransformer([
        ('scaler_credit', KBinsDiscretizer(n_bins=10, encode='ordinal', strategy='quantile'), ['Credit amount'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

X['Age'] = ct_age.fit_transform(_X)['Age']
X['Credit amount'] = ct_credit.fit_transform(_X)['Credit amount']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42, stratify = y)

clf = tree.DecisionTreeClassifier(random_state=42)

# Treinamento
clf.fit(X_train, y_train)

# Teste
y_pred = clf.predict(X_test)

acuracia = accuracy_score(y_test, y_pred)
print(f'A acurácia do modelo foi de {acuracia*100:.2f}%')

A acurácia do modelo foi de 64.00%
